# 02 — Exploratory Data Analysis

Explores the prepared tiles to understand:
1. Class distributions in BGT vs BRT
2. Survival rates (how many BGT polygons make it into BRT)
3. Geometric feature distributions (area, compactness, vertex count)
4. Graph statistics (node count, edge count, degree distribution per tile)

Run **01_data_prep.ipynb** first.


## 0. CONFIG

In [ ]:
from pathlib import Path

CONFIG = {
    "output_root":    Path("processed"),
    "bgt_class_col":  "plus-fysiekVoorkomenWegdeel",   # must match notebook 01
    "brt_class_col":  "verhardingstype",
    "crs":            "EPSG:28992",

    # Number of sample tiles to load for EDA (keep small for speed)
    "eda_n_tiles":    30,

    # Adjacency distance for graph edge construction (metres)
    # Two polygons are connected if they share a boundary or are within this distance
    "adjacency_distance_m": 0.5,
}

## 1. Imports

In [ ]:
import json
import random
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.errors import ShapelyDeprecationWarning
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

random.seed(42)
print("Imports OK")

## 2. Load sample tiles

In [ ]:
with open(CONFIG["output_root"] / "tile_index.json") as f:
    tile_index = json.load(f)

print("Tiles per split:")
for split, tiles in tile_index.items():
    print(f"  {split:5s}: {len(tiles)}")

def load_sample(tile_index, split, n):
    sample = random.sample(tile_index[split], min(n, len(tile_index[split])))
    bgt_frames, brt_frames = [], []
    for t in sample:
        bgt_frames.append(gpd.read_file(t["bgt"]))
        brt_frames.append(gpd.read_file(t["brt"]))
    return (
        gpd.GeoDataFrame(pd.concat(bgt_frames, ignore_index=True)),
        gpd.GeoDataFrame(pd.concat(brt_frames, ignore_index=True)),
    )

n = CONFIG["eda_n_tiles"]
bgt_sample, brt_sample = load_sample(tile_index, "train", n)
print(f"\nLoaded {len(bgt_sample):,} BGT and {len(brt_sample):,} BRT polygons from {n} train tiles")

## 3. Survival rate

In [ ]:
if "survived" in bgt_sample.columns:
    total    = len(bgt_sample)
    survived = bgt_sample["survived"].sum()
    print(f"Overall survival rate: {survived:,} / {total:,} = {survived/total*100:.1f}%")

    # Survival rate per class
    bgt_col = CONFIG["bgt_class_col"]
    if bgt_col in bgt_sample.columns:
        by_class = (
            bgt_sample.groupby(bgt_col)["survived"]
            .agg(["sum", "count"])
            .rename(columns={"sum": "survived", "count": "total"})
        )
        by_class["rate"] = by_class["survived"] / by_class["total"]
        by_class = by_class.sort_values("rate")

        fig, ax = plt.subplots(figsize=(10, max(4, len(by_class) * 0.4)))
        by_class["rate"].plot.barh(ax=ax, color="steelblue")
        ax.axvline(0.5, color="red", linestyle="--", linewidth=1)
        ax.set_title("BGT polygon survival rate by class")
        ax.set_xlabel("Fraction surviving into BRT")
        plt.tight_layout()
        plt.savefig(CONFIG["output_root"] / "survival_by_class.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("\nSurvival rate per class:")
        print(by_class.to_string())
else:
    print("[WARN] 'survived' column not found — re-run notebook 01")

## 4. Geometric feature distributions

In [ ]:
def geom_features(gdf):
    df = pd.DataFrame()
    df["area"]        = gdf.geometry.area
    df["perimeter"]   = gdf.geometry.length
    df["compactness"] = (4 * np.pi * df["area"]) / (df["perimeter"] ** 2 + 1e-9)
    df["n_vertices"]  = gdf.geometry.apply(
        lambda g: len(g.exterior.coords) if g.geom_type == "Polygon"
        else sum(len(p.exterior.coords) for p in g.geoms)
    )
    return df

bgt_geom = geom_features(bgt_sample)
brt_geom = geom_features(brt_sample)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = [
    ("area",        "Area (m²)",        True),
    ("perimeter",   "Perimeter (m)",     True),
    ("n_vertices",  "Vertex count",      True),
    ("compactness", "Compactness",       False),
]
for ax, (col, label, log) in zip(axes.flat, metrics):
    bv = np.log1p(bgt_geom[col]) if log else bgt_geom[col]
    rv = np.log1p(brt_geom[col]) if log else brt_geom[col]
    ax.hist(bv.dropna(), bins=50, alpha=0.6, color="steelblue", label="BGT", density=True)
    ax.hist(rv.dropna(), bins=50, alpha=0.6, color="tomato",    label="BRT", density=True)
    ax.set_title(("log(1+" + label + ")") if log else label)
    ax.legend()

plt.suptitle("BGT vs BRT geometric distributions", fontsize=13)
plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "geometric_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"BGT median area    : {bgt_geom['area'].median():.1f} m²")
print(f"BRT median area    : {brt_geom['area'].median():.1f} m²")
print(f"BGT median vertices: {bgt_geom['n_vertices'].median():.0f}")
print(f"BRT median vertices: {brt_geom['n_vertices'].median():.0f}")
print(f"Polygon count ratio (BRT/BGT): {len(brt_sample)/len(bgt_sample):.3f}")

## 5. Graph statistics

Preview what the GNN graphs will look like — node count, edge count, degree distribution.

In [ ]:
def count_edges(gdf: gpd.GeoDataFrame, dist: float) -> int:
    """
    Count adjacency edges: two polygons are connected if they
    are within `dist` metres of each other.
    """
    buffered = gdf.copy()
    buffered["geometry"] = gdf.geometry.buffer(dist)
    joined = gpd.sjoin(buffered[["geometry"]], gdf[["geometry"]],
                        how="left", predicate="intersects")
    # Remove self-loops
    joined = joined[joined.index != joined["index_right"]]
    return len(joined) // 2  # undirected


sample_tiles = random.sample(tile_index["train"], min(20, len(tile_index["train"])))
stats = []
dist  = CONFIG["adjacency_distance_m"]

for t in sample_tiles:
    bgt_t = gpd.read_file(t["bgt"])
    n_nodes = len(bgt_t)
    n_edges = count_edges(bgt_t, dist)
    survival = bgt_t["survived"].mean() if "survived" in bgt_t.columns else float("nan")
    stats.append({"n_nodes": n_nodes, "n_edges": n_edges,
                  "avg_degree": (2 * n_edges / max(n_nodes, 1)),
                  "survival_rate": survival})

stats_df = pd.DataFrame(stats)
print("Graph statistics per tile (sample of 20):")
print(stats_df.describe().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(stats_df["n_nodes"], bins=15, color="steelblue", edgecolor="white")
axes[0].set_title("Nodes per tile (BGT polygons)")
axes[0].set_xlabel("Count")

axes[1].hist(stats_df["n_edges"], bins=15, color="darkorange", edgecolor="white")
axes[1].set_title("Edges per tile (adjacency)")
axes[1].set_xlabel("Count")

axes[2].hist(stats_df["survival_rate"].dropna(), bins=15, color="seagreen", edgecolor="white")
axes[2].set_title("Survival rate per tile")
axes[2].set_xlabel("Fraction")

plt.suptitle("Graph statistics — train tiles sample", fontsize=13)
plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "graph_statistics.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Visual: one tile with survival labels

In [ ]:
example = tile_index["train"][0]
bgt_ex  = gpd.read_file(example["bgt"])
brt_ex  = gpd.read_file(example["brt"])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# BGT coloured by survival
if "survived" in bgt_ex.columns:
    bgt_ex[bgt_ex["survived"] == 1].plot(ax=axes[0], color="steelblue", alpha=0.7)
    bgt_ex[bgt_ex["survived"] == 0].plot(ax=axes[0], color="tomato",    alpha=0.5)
    axes[0].set_title(f"BGT — blue=survived, red=dropped ({len(bgt_ex)} polygons)")
else:
    bgt_ex.plot(ax=axes[0], color="steelblue")
    axes[0].set_title(f"BGT input ({len(bgt_ex)} polygons)")
axes[0].set_axis_off()

# Overlay
bgt_ex.plot(ax=axes[1], color="steelblue", alpha=0.4)
brt_ex.plot(ax=axes[1], facecolor="none", edgecolor="black", linewidth=1.2)
axes[1].set_title("BGT (blue) + BRT outline (black)")
axes[1].set_axis_off()

# BRT target
brt_ex.plot(ax=axes[2], color="tomato", edgecolor="white", linewidth=0.3)
axes[2].set_title(f"BRT target ({len(brt_ex)} polygons)")
axes[2].set_axis_off()

plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "tile_example.png", dpi=150, bbox_inches="tight")
plt.show()